<a href="https://colab.research.google.com/github/greek-nlp/benchmark/blob/main/gr_retrieval_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Barzokas Retrieval Analysis (Colab-ready)

This notebook runs in Colab and analyzes retrieval setup on Barzokas data:
- **query** = `title`
- **passage** = `text`

It loads the dataset directly from the benchmark source repository, then reports:
- sample titles
- token-level statistics for `title` and `text`
- number of unique authors
- author distribution table and plot

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Colab/local path setup
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    repo_root = Path('/content/benchmark')
    if not repo_root.exists():
        subprocess.run(['git', 'clone', 'https://github.com/greek-nlp/benchmark.git', str(repo_root)], check=True)
    os.chdir(repo_root)
else:
    repo_root = Path.cwd()

print('IN_COLAB:', IN_COLAB)
print('repo_root:', Path.cwd())

In [ ]:
# Load Barzokas source data directly (no data_wrapper dependency)

def run_git(cmd, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)


def load_barzokas_dataframe(root_dir: Path) -> pd.DataFrame:
    registry = pd.read_csv(root_dir / 'data.csv')
    resource_id = 56
    repo_url = registry.loc[registry['id'] == resource_id, 'url'].iloc[0]

    checkout_dir = root_dir / f'repo_{resource_id}'
    if checkout_dir.exists():
        shutil.rmtree(checkout_dir)

    run_git(['git', 'init', str(checkout_dir)])
    run_git(['git', 'remote', 'add', '-f', 'origin', repo_url], cwd=checkout_dir)
    run_git(['git', 'config', 'core.sparseCheckout', 'true'], cwd=checkout_dir)

    sparse_checkout = checkout_dir / '.git' / 'info' / 'sparse-checkout'
    sparse_checkout.write_text('data/corpora
', encoding='utf-8')
    run_git(['git', 'pull', 'origin', 'master'], cwd=checkout_dir)

    data_root_dir = checkout_dir / 'data' / 'corpora'
    df_list = []

    for publisher_dir in sorted(os.listdir(data_root_dir)):
        publisher_path = data_root_dir / publisher_dir
        df_tsv = pd.read_csv(publisher_path / 'metadata.tsv', sep='	')
        rows = []

        txt_root_dir = publisher_path / 'text'
        for status_dir in sorted(os.listdir(txt_root_dir)):
            status_path = txt_root_dir / status_dir
            for txt_file in sorted(os.listdir(status_path)):
                if not txt_file.endswith('.txt'):
                    continue

                txt_id = Path(txt_file).stem
                txt_content = (status_path / txt_file).read_text(encoding='utf-8').replace('
', '')

                row = df_tsv[df_tsv['id'] == txt_id]
                if row.empty:
                    continue

                row_data = row.iloc[0].to_dict()
                row_data['text'] = txt_content
                row_data['status'] = status_dir
                row_data['publisher'] = publisher_dir
                rows.append(row_data)

        if rows:
            df_list.append(pd.DataFrame(rows))

    if not df_list:
        raise RuntimeError('No Barzokas rows were loaded. Check repository structure.')

    df = pd.concat(df_list, ignore_index=True)
    shutil.rmtree(checkout_dir)
    return df


barzokas_df = load_barzokas_dataframe(Path.cwd())
print('Loaded shape:', barzokas_df.shape)
barzokas_df.head(3)

In [ ]:
# Keep retrieval columns
retrieval_df = barzokas_df[['title', 'text', 'author']].copy()
retrieval_df['title'] = retrieval_df['title'].fillna('').astype(str).str.strip()
retrieval_df['text'] = retrieval_df['text'].fillna('').astype(str).str.strip()
retrieval_df['author'] = retrieval_df['author'].fillna('UNKNOWN').astype(str).str.strip()

print('Retrieval rows:', len(retrieval_df))
retrieval_df.head()

In [ ]:
# Sample titles (query examples)
sample_n = min(20, len(retrieval_df))
title_samples = retrieval_df[['title', 'author']].sample(n=sample_n, random_state=42).reset_index(drop=True)
title_samples

In [ ]:
# Token-level statistics (whitespace tokenization)
def token_count(text: str) -> int:
    return len(text.split()) if isinstance(text, str) and text else 0

retrieval_df['title_tokens'] = retrieval_df['title'].apply(token_count)
retrieval_df['text_tokens'] = retrieval_df['text'].apply(token_count)

token_stats = pd.DataFrame({
    'title_tokens': retrieval_df['title_tokens'],
    'text_tokens': retrieval_df['text_tokens'],
}).describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]).T

token_stats['total_tokens'] = [
    retrieval_df['title_tokens'].sum(),
    retrieval_df['text_tokens'].sum(),
]

token_stats

In [ ]:
# Authors: count and distribution
n_authors = retrieval_df['author'].nunique()
print('Unique authors:', n_authors)

author_dist = retrieval_df['author'].value_counts().rename_axis('author').reset_index(name='count')
author_dist['percentage'] = (author_dist['count'] / len(retrieval_df) * 100).round(2)
author_dist.head(30)

In [ ]:
# Author distribution plot (top 25)
top_k = 25
plot_df = author_dist.head(top_k).copy()

plt.figure(figsize=(10, 8))
sns.barplot(data=plot_df, y='author', x='count', color='#2a9d8f')
plt.title(f'Top {top_k} Authors by Number of Passages')
plt.xlabel('Count')
plt.ylabel('Author')
plt.tight_layout()
plt.show()